# RAG Demo: OLG Models in Macroeconomics

This notebook is the single classroom handout for the portable OLG RAG demo. It follows the same style as the Ren Zhengfei RAG demo:

```text
source document -> chunks -> TF-IDF index -> retrieval audit -> grounded answer
```

The default path is fully local and deterministic. It uses a standard-library TF-IDF retriever and a local extractive answerer, so students can run every core step without API keys, network access, vector databases, or neural embeddings.

The optional Claude Code cell near the end can compare a naive LLM answer with a retrieval-grounded answer, but it is off by default.


## Step 0: Setup

Run this notebook from the `rag_olg_demo` folder. The source markdown is already copied into `data/source/`, so the demo is portable.


In [1]:
from pathlib import Path
import csv
import json
import sys

ROOT = Path.cwd()
if not (ROOT / "run.py").exists() or not (ROOT / "src" / "rag_olg").exists():
    raise RuntimeError("Please launch this notebook from the rag_olg_demo folder.")

sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))

from questions import QUESTIONS
from rag_olg.chunking import split_markdown_into_chunks
from rag_olg.generation import build_prompt, extractive_answer
from rag_olg.llm import claude_code_auth_check, compare, hits_anchor
from rag_olg.retrieval import build_tfidf_index, search

SOURCE = ROOT / "data" / "source" / "Spear-Young_OLG_final_preprint.md"
BUILD_DIR = ROOT / "build"
BUILD_DIR.mkdir(exist_ok=True)

print(f"Demo root: {ROOT}")
print(f"Source:    {SOURCE.relative_to(ROOT)}")
print(f"Exists:    {SOURCE.exists()}")


Demo root: /Users/zfeng/Library/CloudStorage/OneDrive-Personal/Teachings/AI_ML/Lecture_notes_2026/2026_New_Slides/MiniCourse_8hr/demos/rag_olg_demo
Source:    data/source/Spear-Young_OLG_final_preprint.md
Exists:    True


## Step 1: Source Audit

Before retrieving anything, inspect the corpus. Here the corpus is a single paper converted to markdown, with line numbers preserved by the chunker for citation.


In [2]:
markdown = SOURCE.read_text(encoding="utf-8")
lines = markdown.splitlines()
headings = [line for line in lines if line.startswith("#")]

print(f"Characters: {len(markdown):,}")
print(f"Lines:      {len(lines):,}")
print(f"Headings:   {len(headings):,}")
print("\nFirst five headings:")
for heading in headings[:5]:
    print(" ", heading[:110])


Characters: 726,287
Lines:      3,786
Headings:   56

First five headings:
  # **Overlapping**
  # **Generations**
  # **Table of Contents**
  # **Acknowledgements**
  # **Introduction**


## Step 2: Chunk The Paper

The chunker preserves source line ranges and recent markdown headings. This is why retrieved evidence can cite `data/source/...:Lx-Ly` instead of becoming anonymous text.


In [3]:
MAX_CHARS = 2200
OVERLAP_LINES = 8

chunks = split_markdown_into_chunks(
    markdown,
    source_name="data/source/Spear-Young_OLG_final_preprint.md",
    max_chars=MAX_CHARS,
    overlap_lines=OVERLAP_LINES,
)

print(f"Chunks:        {len(chunks)}")
print(f"Max chars:     {MAX_CHARS}")
print(f"Overlap lines: {OVERLAP_LINES}")
print("\nFirst three chunks:")
for chunk in chunks[:3]:
    print(f"  {chunk.chunk_id} | {chunk.citation} | {chunk.section}")


Chunks:        525
Max chars:     2200
Overlap lines: 8

First three chunks:
  olg-0001 | data/source/Spear-Young_OLG_final_preprint.md:L1-L42 | Overlapping
  olg-0002 | data/source/Spear-Young_OLG_final_preprint.md:L35-L52 | Acknowledgements
  olg-0003 | data/source/Spear-Young_OLG_final_preprint.md:L46-L56 | Introduction


## Step 3: Build A Local TF-IDF Index

This is not neural embedding retrieval. It is an intentionally transparent sparse-vector baseline:

```text
chunk text -> tokens -> TF-IDF weights -> cosine similarity
```

That is enough to teach the architecture before swapping in production embeddings.


In [4]:
index = build_tfidf_index(chunks)

print(f"TF-IDF vocabulary size: {len(index.idf):,}")
print(f"Indexed vectors:        {len(index.vectors):,}")
print(f"Example terms:          {list(index.idf)[:12]}")


TF-IDF vocabulary size: 55,470
Indexed vectors:        525
Example terms:          ['workhorse', 'solg', '91_chapter', '154_chapter', 'ascendance', 'recording', 'olg_theory', 'overlapping', 'sara_put', 'models', '19', '250']


## Step 4: Curated Classroom Questions

The questions are designed to expose the difference between a generic graduate macro answer and the specific historical argument in Spear and Young's paper.


In [5]:
try:
    from IPython.display import Markdown, display
    rows = ["| # | Theme | Question |", "|---|-------|----------|"]
    for item in QUESTIONS:
        short = item["q"][:82] + ("..." if len(item["q"]) > 82 else "")
        rows.append(f"| {item['id']} | {item['theme']} | {short} |")
    display(Markdown("\n".join(rows)))
except Exception:
    for item in QUESTIONS:
        print(f"{item['id']}. {item['theme']} - {item['q']}")


| # | Theme | Question |
|---|-------|----------|
| 1 | OLG vs ILA | How do Spear and Young distinguish overlapping-generations models from infinite-li... |
| 2 | Samuelson 1958 | Why was Samuelson 1958 important for monetary economics and the OLG tradition? |
| 3 | Diamond 1965 | What did Diamond 1965 add to the OLG approach? |
| 4 | Lucas 1972 | What role did Lucas 1972 play in the paper's history of OLG models? |
| 5 | ILA Ascendance | Why do the authors argue ILA became dominant over OLG? |
| 6 | Stochastic OLG | What does the paper say about stochastic OLG models and recursive equilibrium? |
| 7 | Textbook Coverage | What does the paper say about graduate textbook coverage of OLG models? |
| 8 | Unsupported Query | What does this paper say about the December 2025 FOMC meeting? |

## Step 5: Retrieval Diagnostics Before Generation

Do not start with the LLM. First check whether retrieval found plausible evidence. For supported questions, at least one anchor phrase should appear in the top chunks. For the unsupported question, the extractive answer should refuse.


In [6]:
diagnostics = []
for item in QUESTIONS:
    retrieved = search(chunks, index, item["q"], top_k=5)
    answer = extractive_answer(item["q"], retrieved)
    anchor_hits = hits_anchor(retrieved, item.get("anchors", []), top_n=5)
    refused = "do not have enough retrieved evidence" in answer
    ok = refused if item.get("expect_refusal") else (retrieved[0].score > 0.035 and len(anchor_hits) >= 1)
    diagnostics.append({
        "id": item["id"],
        "theme": item["theme"],
        "top_score": retrieved[0].score,
        "anchor_hits": anchor_hits,
        "expected_refusal": item.get("expect_refusal", False),
        "refused": refused,
        "ok": ok,
        "top_sources": [r.chunk.citation for r in retrieved],
    })
    status = "PASS" if ok else "FAIL"
    if item.get("expect_refusal"):
        print(f"{status} | Q{item['id']:02d} | refusal={refused} | {item['theme']}")
    else:
        print(f"{status} | Q{item['id']:02d} | score={retrieved[0].score:.3f} | anchors={len(anchor_hits)}/{len(item.get('anchors', []))} | {item['theme']}")

print(f"\nAll diagnostics passed: {all(row['ok'] for row in diagnostics)}")


PASS | Q01 | score=0.058 | anchors=2/3 | OLG vs ILA
PASS | Q02 | score=0.067 | anchors=3/3 | Samuelson 1958
PASS | Q03 | score=0.093 | anchors=3/3 | Diamond 1965
PASS | Q04 | score=0.102 | anchors=3/3 | Lucas 1972
PASS | Q05 | score=0.135 | anchors=4/4 | ILA Ascendance
PASS | Q06 | score=0.116 | anchors=3/3 | Stochastic OLG
PASS | Q07 | score=0.157 | anchors=3/3 | Textbook Coverage
PASS | Q08 | refusal=True | Unsupported Query

All diagnostics passed: True


## Step 6: Single-Question Walkthrough

Change `QUESTION_ID` to explore another topic. The default uses Diamond 1965 because it produces a clean, compact evidence trail.


In [7]:
QUESTION_ID = 3
TOP_K = 5

question_item = next(item for item in QUESTIONS if item["id"] == QUESTION_ID)
question = question_item["q"]
retrieved = search(chunks, index, question, top_k=TOP_K)

print(f"Q{QUESTION_ID}: {question_item['theme']}")
print(question)
print("\nTop retrieved chunks:\n")
for rank, result in enumerate(retrieved, 1):
    chunk = result.chunk
    preview = " ".join(chunk.text.split())[:360]
    print(f"S{rank}: score={result.score:.3f} | {chunk.citation}")
    print(f"    Section: {chunk.section}")
    print(f"    {preview}...\n")

print("Anchor hits:", hits_anchor(retrieved, question_item.get("anchors", []), top_n=TOP_K))


Q3: Diamond 1965
What did Diamond 1965 add to the OLG approach?

Top retrieved chunks:

S1: score=0.093 | data/source/Spear-Young_OLG_final_preprint.md:L3048-L3054
    Section: Chapter 11: Summary and Conclusion
    In order to present a coherent conclusion to the mass of material presented in the book, we utilized the morphological approach outlined in the previous chapter. We can identify two essential model types: foundational and augmented, in addition to four areas of model efficacy: theoretical, computational, policy, and pedagogical (see for example, Jordi [2018,...

S2: score=0.092 | data/source/Spear-Young_OLG_final_preprint.md:L368-L374
    Section: Chapter 2: OLG – The Next Generations, 1960-1970 > Section 2.3 The Diamond Model
    Diamond then looks at the effect of introducing different types of government debt and shows that if the economy was originally operating efficiently (i.e., where interest rates are greater than or equal to the population growth rate), debt reduce

## Step 7: Inspect The Grounded Prompt

The prompt is the contract between retrieval and generation. It tells the answerer to use only retrieved context and to cite `[S1]`, `[S2]`, etc.


In [8]:
prompt = build_prompt(question, retrieved)
print(prompt[:3500])
print("\n... prompt truncated for display ...")


You are answering a macroeconomics question using retrieved evidence from Spear and Young's OLG paper.

Rules:
- Answer only from the retrieved context.
- Cite every substantive claim with source labels such as [S1] or [S2].
- If the retrieved context is insufficient, say so directly.
- Do not use outside knowledge.

Retrieved context:
[S1] data/source/Spear-Young_OLG_final_preprint.md:L3048-L3054
Section: Chapter 11: Summary and Conclusion
Retrieval score: 0.093
In order to present a coherent conclusion to the mass of material presented in the book, we utilized the morphological approach outlined in the previous chapter. We can identify two essential model types: foundational and augmented, in addition to four areas of model efficacy: theoretical, computational, policy, and pedagogical (see for example, Jordi [2018, p. 87]). Given this, we propose that the plasticity of the two respective approaches enabled their metamorphosis from foundational to augmented types.

Now, the formalized

## Step 8: Local Extractive Answer

This answer is not trying to be eloquent. It is deliberately auditable: each sentence is selected from retrieved chunks and labeled with an `S` citation.


In [9]:
local_answer = extractive_answer(question, retrieved)
print(local_answer)


Before leaving Diamond's 1965 approach, we look at the development of his OLG approach over the period 1964-1973. [S2] A few years later, Peter Diamond (1965) reworked the OLG approach by introducing production in the model. [S3] Now, the formalized canonical foundational OLG model of Samuelson (1958) took almost a decade to have an impact, and this came via the contribution of Diamond (1965). [S1] Canonical augmentations of the OLG approach were the outcome of the work of Auerbach and Kotlikoff from the mid-1980s onwards, and, for the ILA model, the parallel development of the macroeconomic DSGE approach, of both New Classical and New Keynesian vintage, with RBC at its core. [S1]


## Step 9: Optional Claude Code Naive-vs-RAG Comparison

Default: skipped. Set `ENABLE_LLM = True` to call local `claude -p`. This requires Claude Code to be installed and logged in, but no Anthropic API key is needed.


In [10]:
ENABLE_LLM = False
LLM_MODEL = "sonnet"

if ENABLE_LLM:
    ok, msg = claude_code_auth_check(timeout=30)
    print(msg)
    if not ok:
        raise RuntimeError("Claude Code is not ready. Run `claude` in a terminal and use /login.")
    llm_result = compare(question, chunks, index, top_k=TOP_K, model=LLM_MODEL)
    print("\n----- Naive Claude, no retrieval -----\n")
    print(llm_result["naive"])
    print("\n----- RAG Claude, retrieved context only -----\n")
    print(llm_result["rag"])
else:
    print("Skipped. Set ENABLE_LLM = True to run the optional Claude Code comparison.")


Skipped. Set ENABLE_LLM = True to run the optional Claude Code comparison.


## Step 10: Refusal Demo

RAG should also know when not to answer. This query is outside the paper, so the local extractive answer should refuse rather than hallucinate.


In [11]:
unsupported = next(item for item in QUESTIONS if item.get("expect_refusal"))
unsupported_results = search(chunks, index, unsupported["q"], top_k=5)
print(unsupported["q"])
print("\nTop score:", f"{unsupported_results[0].score:.3f}")
print("\nAnswer:")
print(extractive_answer(unsupported["q"], unsupported_results))


What does this paper say about the December 2025 FOMC meeting?

Top score: 0.060

Answer:
I do not have enough retrieved evidence from the OLG paper to answer that question. Try a question using terms from the paper, such as OLG, ILA, Samuelson, Diamond, Lucas, Barro, or stochastic OLG.


## Step 11: Write A Retrieval Report

The notebook writes a small CSV report to `build/` so instructors can check retrieval stability after editing questions or chunk settings.


In [12]:
report_path = BUILD_DIR / "notebook_retrieval_report.csv"
with report_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["id", "theme", "top_score", "anchor_hits", "expected_refusal", "refused", "ok", "top_sources"],
    )
    writer.writeheader()
    for row in diagnostics:
        writer.writerow({
            "id": row["id"],
            "theme": row["theme"],
            "top_score": f"{row['top_score']:.6f}",
            "anchor_hits": "; ".join(row["anchor_hits"]),
            "expected_refusal": row["expected_refusal"],
            "refused": row["refused"],
            "ok": row["ok"],
            "top_sources": "; ".join(row["top_sources"]),
        })
print(f"Wrote: {report_path.relative_to(ROOT)}")


Wrote: build/notebook_retrieval_report.csv


## Discussion Prompts

1. What changed between the generic macro answer and the grounded RAG answer?
2. Which step is most fragile: chunking, retrieval, prompt construction, or generation?
3. Why does line-addressable citation matter in a research workflow?
4. What would neural embeddings improve, and what would they not fix?
5. When should a RAG system refuse to answer?
